# Module 2: Evaluation & Baseline Testing

## Overview

In this module, you'll establish a baseline for your multi-agent system by:

1. **Running a comprehensive evaluation dataset** with 25+ test cases
2. **Using built-in evaluators** from strands-evals
3. **Creating custom evaluators** for domain-specific metrics
4. **Establishing baseline metrics** for production monitoring

### Learning Objectives
1. Understand evaluation strategies for multi-agent systems
2. Use strands-evals for automated evaluation
3. Build custom evaluators for business requirements
4. Establish metrics baselines for production

### Time: ~60 minutes

## Step 1: Setup

Install dependencies and create the customer service instance.

In [2]:
# Install dependencies
!pip install -q strands-agents strands-agents-evals boto3 pandas


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [3]:
import json
import os
import sys
import boto3
import pandas as pd
from datetime import datetime

# Try to load REGION from Module 1
try:
    %store -r REGION
    print(f"✓ Loaded REGION from Module 1: {REGION}")
except:
    print("Could not load REGION from Module 1, using session default")
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'

print(f"Region: {REGION}")

# Set environment variable
os.environ['AWS_REGION'] = REGION

# Always create new customer_service instance (cannot be pickled)
print("\nCreating customer service instance...")
sys.path.insert(0, '../01-multi-agent-prototype/agents')
from orchestrator import MultiAgentCustomerService
customer_service = MultiAgentCustomerService(region=REGION)
print("✓ Customer service initialized")

✓ Loaded REGION from Module 1: us-west-2
Region: us-west-2

Creating customer service instance...
✓ Customer service initialized


## Step 2: Load Evaluation Experiment

Load the test cases and convert them to the format expected by strands-evals.

In [4]:
# Import strands-evals
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

# Load evaluation dataset
with open('evaluation_dataset.json', 'r') as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data['test_cases'])} test cases")
print(f"\nSample test case:")
print(json.dumps(eval_data['test_cases'][0], indent=2))

Loaded 25 test cases

Sample test case:
{
  "id": "order_001",
  "category": "order_status",
  "difficulty": "easy",
  "input": "What's the status of order ORD-2024-10002?",
  "expected_agent": "order_agent",
  "expected_tools": [
    "get_order_status"
  ],
  "expected_output_contains": [
    "shipped",
    "UPS",
    "tracking"
  ],
  "ground_truth": "Order ORD-2024-10002 is shipped via UPS with tracking number 1Z999AA10123456784"
}


In [5]:
# Convert test cases to Case objects
test_cases = []
for tc in eval_data['test_cases']:
    case = Case(
        name=tc['id'],
        input=tc['input'],
        expected_output=tc.get('ground_truth', ''),
        metadata={
            'category': tc['category'],
            'expected_agent': tc.get('expected_agent'),
            'expected_tools': tc.get('expected_tools', []),
            'expected_output_contains': tc.get('expected_output_contains', [])
        }
    )
    test_cases.append(case)

print(f"Created {len(test_cases)} Case objects")
print(f"\nCategories: {set(tc.metadata['category'] for tc in test_cases)}")

Created 25 Case objects

Categories: {'order_history', 'order_status', 'account_info', 'product_inventory', 'product_search', 'product_comparison', 'membership_benefits', 'multi_agent', 'suspended_account', 'order_cancellation', 'order_tracking', 'product_recommendation', 'product_policy', 'payment_methods', 'password_reset', 'order_return', 'edge_case', 'refund_status', 'order_not_found', 'product_details', 'address_update', 'out_of_scope'}


## Step 3: Define Task Function

Create a function that runs the agent on each test case.

In [6]:
def run_customer_service_task(case: Case) -> str:
    """
    Task function that runs a test case through the multi-agent system.
    
    Args:
        case: Test case to run
        
    Returns:
        str: Agent response
    """
    try:
        response = customer_service.chat(case.input)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"

print("Task function defined")
print("\nTesting task function with one case...")
test_response = run_customer_service_task(test_cases[0])
print(f"Response: {test_response[:200]}...")

Task function defined

Testing task function with one case...

Tool #1: delegate_to_order_agent

Tool #1: get_order_status
Great! Here's the status of your order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number provided above. Would you like me to get more detailed tracking information or help you with anything else?Great news! Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99

## Step 4: Run Evaluation with Built-in Evaluators

Use OutputEvaluator to evaluate goal success and helpfulness.

**Note:** Each Experiment can only have ONE evaluator, so we create separate experiments for each metric.

In [7]:
# Create Goal Success Evaluator
goal_success_evaluator = OutputEvaluator(
    rubric="""Evaluate if the agent response successfully addresses the customer's request.

Score 1.0: The response fully addresses the customer's request with accurate, helpful information
Score 0.75: The response mostly addresses the request but may be missing minor details
Score 0.5: The response partially addresses the request but has significant gaps
Score 0.25: The response attempts to address the request but fails to provide useful help
Score 0.0: The response does not address the request at all or provides incorrect information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

print("Goal Success Evaluator created")

Goal Success Evaluator created


In [8]:
# Run Goal Success Evaluation (using subset for demo)
# Remove [:5] to run all test cases
goal_success_experiment = Experiment(
    cases=test_cases[:5],  # First 5 test cases for demo
    evaluators=[goal_success_evaluator]  # Pass as list
)

print("Running goal success evaluation...")
goal_success_results = goal_success_experiment.run_evaluations(run_customer_service_task)

Running goal success evaluation...

Tool #2: delegate_to_order_agent

Tool #2: get_order_status
Here's the status of your order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:

In [11]:


# Extract the report (first and only item in results list)
goal_success_report = goal_success_results[0]

print("\n" + "="*60)
print("GOAL SUCCESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], goal_success_report.scores, goal_success_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Goal Success Score: {goal_success_report.overall_score:.2f}")
print(f"Pass Rate: {sum(goal_success_report.test_passes)}/{len(goal_success_report.test_passes)}")
print(f"{'='*60}")


GOAL SUCCESS EVALUATION RESULTS

1. order_001 (order_status)
   Score: 1.00
   Reasoning: The output fully addresses the customer's order status request. It provides the essential information (shipped status, UPS carrier, tracking number 1Z...

2. order_002 (order_tracking)
   Score: 1.00
   Reasoning: The output fully addresses the customer's tracking request with all key information: carrier (DHL), tracking number (DHL1234567890), and estimated del...

3. order_003 (order_return)
   Score: 0.25
   Reasoning: The output addresses the customer's return request but provides incorrect information. The expected output indicates the return should be successful f...

4. order_004 (order_cancellation)
   Score: 0.00
   Reasoning: The output provides completely incorrect information. It states the order is already cancelled when the expected output indicates the order is pending...

5. order_005 (order_history)
   Score: 1.00
   Reasoning: The output fully addresses the customer's request by

In [12]:
# Create Helpfulness Evaluator
helpfulness_evaluator = OutputEvaluator(
    rubric="""Evaluate how helpful the agent response is for the customer.

Score 1.0: Extremely helpful - provides clear, actionable information and anticipates follow-up needs
Score 0.75: Very helpful - provides good information that addresses the customer's needs
Score 0.5: Somewhat helpful - provides basic information but could be more detailed
Score 0.25: Minimally helpful - provides limited useful information
Score 0.0: Not helpful - does not provide any useful information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

# Run Helpfulness Evaluation
helpfulness_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[helpfulness_evaluator]  # Pass as list
)

print("Running helpfulness evaluation...")
helpfulness_results = helpfulness_experiment.run_evaluations(run_customer_service_task)

# Extract the report
helpfulness_report = helpfulness_results[0]

print("\n" + "="*60)
print("HELPFULNESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], helpfulness_report.scores, helpfulness_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Helpfulness Score: {helpfulness_report.overall_score:.2f}")
print(f"Pass Rate: {sum(helpfulness_report.test_passes)}/{len(helpfulness_report.test_passes)}")
print(f"{'='*60}")

Running helpfulness evaluation...

Tool #7: delegate_to_order_agent

Tool #7: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA101234

## Step 5: Custom Evaluators

Run domain-specific custom evaluators for routing accuracy, policy compliance, and response quality.

In [31]:
  # Reload custom evaluators module (in case it was already imported)
  import importlib
  import sys
  if 'custom_evaluators' in sys.modules:
      import custom_evaluators
      importlib.reload(custom_evaluators)

In [32]:
# Import custom evaluators
from custom_evaluators import (
    RoutingAccuracyEvaluator,
    PolicyComplianceEvaluator,
    ResponseQualityEvaluator,
    CustomerSatisfactionEvaluator
)

print("Custom evaluators imported")

Custom evaluators imported


In [33]:
# Routing Accuracy Evaluation
routing_evaluator = RoutingAccuracyEvaluator()
routing_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[routing_evaluator]  # Pass as list
)

print("Running routing accuracy evaluation...")
routing_results = routing_experiment.run_evaluations(run_customer_service_task)

# Extract the report
routing_report = routing_results[0]

print("\n" + "="*60)
print("ROUTING ACCURACY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], routing_report.scores, routing_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Routing Accuracy Score: {routing_report.overall_score:.2f}")
print(f"Pass Rate: {sum(routing_report.test_passes)}/{len(routing_report.test_passes)}")
print(f"{'='*60}")

Running routing accuracy evaluation...

Tool #12: delegate_to_order_agent

Tool #12: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999A

In [34]:
# Policy Compliance Evaluation
policy_evaluator = PolicyComplianceEvaluator()
policy_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[policy_evaluator]  # Pass as list
)

print("Running policy compliance evaluation...")
policy_results = policy_experiment.run_evaluations(run_customer_service_task)

# Extract the report
policy_report = policy_results[0]

print("\n" + "="*60)
print("POLICY COMPLIANCE EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], policy_report.scores, policy_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Policy Compliance Score: {policy_report.overall_score:.2f}")
print(f"Pass Rate: {sum(policy_report.test_passes)}/{len(policy_report.test_passes)}")
print(f"{'='*60}")

Running policy compliance evaluation...

Tool #17: delegate_to_order_agent

Tool #17: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999

In [35]:
# Response Quality Evaluation
quality_evaluator = ResponseQualityEvaluator()
quality_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[quality_evaluator]  # Pass as list
)

print("Running response quality evaluation...")
quality_results = quality_experiment.run_evaluations(run_customer_service_task)

# Extract the report
quality_report = quality_results[0]

print("\n" + "="*60)
print("RESPONSE QUALITY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], quality_report.scores, quality_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Response Quality Score: {quality_report.overall_score:.2f}")
print(f"Pass Rate: {sum(quality_report.test_passes)}/{len(quality_report.test_passes)}")
print(f"{'='*60}")

Running response quality evaluation...

Tool #21: delegate_to_order_agent

Tool #21: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999A

In [36]:
# Customer Satisfaction Evaluation
satisfaction_evaluator = CustomerSatisfactionEvaluator()
satisfaction_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[satisfaction_evaluator]  # Pass as list
)

print("Running customer satisfaction evaluation...")
satisfaction_results = satisfaction_experiment.run_evaluations(run_customer_service_task)

# Extract the report
satisfaction_report = satisfaction_results[0]

print("\n" + "="*60)
print("CUSTOMER SATISFACTION EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], satisfaction_report.scores, satisfaction_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Customer Satisfaction Score: {satisfaction_report.overall_score:.2f}")
print(f"Pass Rate: {sum(satisfaction_report.test_passes)}/{len(satisfaction_report.test_passes)}")
print(f"{'='*60}")

Running customer satisfaction evaluation...
Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your package is on its way and should arrive by January 8, 2025. You can track it using the UPS tracking number above.

Is there anything else you'd like to know?
Tool #25: delegate_to_order_agent

Tool #25: track_shipment
Here's the tracking information for your order **ORD-2024-10009**:

**Shipment Status:** 🚚 **Shipped**

**Tracking Details:**
- **Carrier:** DHL
- **Tracking Number:** DHL1234567890
- **Estimated Delivery:** January 10, 2025
- **Track Your Package:** https://www.dhl.com/en/express/tracking.html?AWB=D

## Step 6: Extract and Analyze Results

Collect scores from all evaluations and compute baseline metrics.

In [37]:
# Helper function to extract scores from EvaluationReport
def extract_scores_from_report(report):
    """Extract scores from evaluation report"""
    return report.scores

# Extract all scores
goal_success_scores = extract_scores_from_report(goal_success_report)
helpfulness_scores = extract_scores_from_report(helpfulness_report)
routing_scores = extract_scores_from_report(routing_report)
policy_scores = extract_scores_from_report(policy_report)
quality_scores = extract_scores_from_report(quality_report)
satisfaction_scores = extract_scores_from_report(satisfaction_report)

print("✓ Scores extracted from all evaluations")
print(f"\nGoal Success: {goal_success_scores}")
print(f"Helpfulness: {helpfulness_scores}")
print(f"Routing: {routing_scores}")
print(f"Policy: {policy_scores}")
print(f"Quality: {quality_scores}")
print(f"Satisfaction: {satisfaction_scores}")

✓ Scores extracted from all evaluations

Goal Success: [1.0, 1.0, 0.25, 0.0, 1.0]
Helpfulness: [1.0, 1.0, 0.75, 0.0, 1.0]
Routing: [1.0, 1.0, 1.0, 1.0, 1.0]
Policy: [1.0, 1.0, 1.0, 0.0, 1.0]
Quality: [1.0, 0.9, 0.2, 0.2, 0.8]
Satisfaction: [1.0, 1.0, 0.25, 0.75, 1.0]


In [38]:
# Create DataFrame with all results
results_df = pd.DataFrame({
    'test_case': [case.name for case in test_cases[:5]],
    'category': [case.metadata['category'] for case in test_cases[:5]],
    'goal_success': goal_success_scores,
    'helpfulness': helpfulness_scores,
    'routing_accuracy': routing_scores,
    'policy_compliance': policy_scores,
    'response_quality': quality_scores,
    'customer_satisfaction': satisfaction_scores
})

print("\nEvaluation Results DataFrame:")
print(results_df.to_string(index=False))


Evaluation Results DataFrame:
test_case           category  goal_success  helpfulness  routing_accuracy  policy_compliance  response_quality  customer_satisfaction
order_001       order_status          1.00         1.00               1.0                1.0               1.0                   1.00
order_002     order_tracking          1.00         1.00               1.0                1.0               0.9                   1.00
order_003       order_return          0.25         0.75               1.0                1.0               0.2                   0.25
order_004 order_cancellation          0.00         0.00               1.0                0.0               0.2                   0.75
order_005      order_history          1.00         1.00               1.0                1.0               0.8                   1.00


## Step 7: Baseline Metrics

Calculate average scores to establish baseline performance.

In [39]:
# Calculate baseline metrics
baseline_metrics = {
    'timestamp': datetime.now().isoformat(),
    'total_test_cases': len(results_df),
    'goal_success_rate': results_df['goal_success'].mean(),
    'helpfulness': results_df['helpfulness'].mean(),
    'routing_accuracy': results_df['routing_accuracy'].mean(),
    'policy_compliance': results_df['policy_compliance'].mean(),
    'response_quality': results_df['response_quality'].mean(),
    'customer_satisfaction': results_df['customer_satisfaction'].mean()
}

print("\n" + "="*60)
print("BASELINE METRICS")
print("="*60)
for metric, value in baseline_metrics.items():
    if isinstance(value, float) and value <= 1.0:
        print(f"{metric:.<40} {value:.1%}")
    else:
        print(f"{metric:.<40} {value}")


BASELINE METRICS
timestamp............................... 2026-01-09T20:21:47.450509
total_test_cases........................ 5
goal_success_rate....................... 65.0%
helpfulness............................. 75.0%
routing_accuracy........................ 100.0%
policy_compliance....................... 80.0%
response_quality........................ 62.0%
customer_satisfaction................... 80.0%


In [40]:
# Performance by category
print("\n" + "="*60)
print("PERFORMANCE BY CATEGORY")
print("="*60)

category_metrics = results_df.groupby('category')[[
    'goal_success', 'helpfulness', 'routing_accuracy', 
    'policy_compliance', 'response_quality', 'customer_satisfaction'
]].mean()

print(category_metrics.to_string())


PERFORMANCE BY CATEGORY
                    goal_success  helpfulness  routing_accuracy  policy_compliance  response_quality  customer_satisfaction
category                                                                                                                   
order_cancellation          0.00         0.00               1.0                0.0               0.2                   0.75
order_history               1.00         1.00               1.0                1.0               0.8                   1.00
order_return                0.25         0.75               1.0                1.0               0.2                   0.25
order_status                1.00         1.00               1.0                1.0               1.0                   1.00
order_tracking              1.00         1.00               1.0                1.0               0.9                   1.00


## Step 8: Define Production Thresholds

Set alert thresholds based on baseline metrics (typically 80-90% of baseline).

In [41]:
# Define production thresholds
production_thresholds = {
    'goal_success_rate': 0.70,       # Alert if below 70%
    'helpfulness': 0.65,              # Alert if below 65%
    'routing_accuracy': 0.75,         # Alert if below 75%
    'policy_compliance': 0.80,        # Alert if below 80%
    'response_quality': 0.65,         # Alert if below 65%
    'customer_satisfaction': 0.70     # Alert if below 70%
}

print("\n" + "="*60)
print("PRODUCTION THRESHOLDS")
print("="*60)
for metric, threshold in production_thresholds.items():
    print(f"{metric:.<40} {threshold:.0%}")


PRODUCTION THRESHOLDS
goal_success_rate....................... 70%
helpfulness............................. 65%
routing_accuracy........................ 75%
policy_compliance....................... 80%
response_quality........................ 65%
customer_satisfaction................... 70%


In [42]:
# Check current performance against thresholds
print("\n" + "="*60)
print("THRESHOLD VALIDATION")
print("="*60)

all_pass = True
for metric, threshold in production_thresholds.items():
    current = baseline_metrics.get(metric, 0)
    passed = current >= threshold
    
    status = "✓ PASS" if passed else "✗ FAIL"
    
    if not passed:
        all_pass = False
    
    print(f"[{status}] {metric}: {current:.1%} (threshold: {threshold:.0%})")

print("\n" + "="*60)
if all_pass:
    print("✓ ALL THRESHOLDS PASSED - Ready for production!")
else:
    print("⚠ SOME THRESHOLDS FAILED - Review before production")
print("="*60)


THRESHOLD VALIDATION
[✗ FAIL] goal_success_rate: 65.0% (threshold: 70%)
[✓ PASS] helpfulness: 75.0% (threshold: 65%)
[✓ PASS] routing_accuracy: 100.0% (threshold: 75%)
[✓ PASS] policy_compliance: 80.0% (threshold: 80%)
[✗ FAIL] response_quality: 62.0% (threshold: 65%)
[✓ PASS] customer_satisfaction: 80.0% (threshold: 70%)

⚠ SOME THRESHOLDS FAILED - Review before production


## Step 9: Save Results

Store evaluation results and baseline metrics for Module 3.

In [43]:
# Save detailed results
results_df.to_csv('evaluation_results.csv', index=False)
print("✓ Saved detailed results to evaluation_results.csv")

# Save baseline metrics
with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2, default=str)
print("✓ Saved baseline metrics to baseline_metrics.json")

# Store for next module
%store baseline_metrics
%store production_thresholds
%store REGION
print("\n✓ Data stored for Module 3!")

✓ Saved detailed results to evaluation_results.csv
✓ Saved baseline metrics to baseline_metrics.json
Stored 'baseline_metrics' (dict)
Stored 'production_thresholds' (dict)
Stored 'REGION' (str)

✓ Data stored for Module 3!


## Summary

In this module, you:

1. ✅ **Loaded evaluation experiment** with 25+ test cases
2. ✅ **Used built-in evaluators** (Goal Success, Helpfulness) 
3. ✅ **Applied custom evaluators** (Routing, Policy, Quality, Satisfaction)
4. ✅ **Established baseline metrics** for production monitoring
5. ✅ **Defined production thresholds** for alerting

### Key Takeaways

- **Each Experiment requires ONE evaluator** - run separate evaluations for multiple metrics
- **Use Case objects** with proper structure (name, input, expected_output, metadata)
- **Task functions** should be simple wrappers that run your agent
- **OutputEvaluator with rubrics** is perfect for domain-specific quality metrics
- **Baseline metrics** help you detect performance degradation in production

### Next Steps

In **Module 3**, we'll deploy this system to production with AgentCore Runtime and configure observability for continuous monitoring.